## Streaming Scores — Global vs. Adaptive Scores

Scores the streaming set using both the global baseline and the EWMA-adaptive baseline, scoring every transaction with each and recording the two scores side by side. This notebook does **not** decide how to combine the two scores into a single fraud decision, and doesn't introduce formal evaluation metrics. The goal here is to check the adaptive scores and produce the score table to be used later.

### Loading data

The global baseline is refit here from `historical.csv`, filtered to non-fraud (`Class == 0`) transactions only.

`streaming.csv` is **not** stored in chronological order, so it's sorted by `Time` before being used since the EWMA update depends on the real elapsed time between consecutive transactions, which only makes sense if they're processed in the order they actually occurred.

In [1]:
import sys
import pandas as pd
import numpy as np

sys.path.append('..')
from app.scoring import MahalanobisScorer

streampath = "../data/streaming.csv"
histpath = "../data/historical.csv"
baseline = pd.read_csv(histpath)
baseline = baseline.loc[baseline.Class == 0]
baseline = baseline.loc[:, ['V' + str(i) for i in range(1, 29, 1)]].to_numpy()

data = pd.read_csv(streampath)
sorted_data = data.sort_values(by='Time').reset_index(drop=True)



### Replaying the stream

The adaptive scorer is warm-started from the fitted global scorer (`fit_ewma`), so at the start of the replay it's identical to the global baseline and only diverges as it adapts.

For each transaction, in chronological order: score it against both the global and adaptive baselines *before* updating anything, so a transaction is never compared against a baseline that already includes itself. The adaptive scorer is only updated on non-fraud transactions (`Class == 0`) since we want the baseline to represent what the normal transactions look like. We are able to do this since our streaming set has fraud labels, but we wouldn't have this luxury on real data.

The very first transaction has no prior streaming transaction to measure elapsed time from, so `elapsed_time` is set to `0` for that one step. Since `w = 0` in that case, it has no effect on the baseline, and the adaptive scorer effectively starts adapting from the second transaction onward.

In [2]:
global_scorer = MahalanobisScorer.fit(baseline)
adaptive_scorer = MahalanobisScorer.fit_ewma(global_scorer)

features = sorted_data.loc[:, ['V' + str(i) for i in range(1, 29, 1)]].to_numpy()
times = sorted_data.loc[:, 'Time']
labels = sorted_data.loc[:, 'Class']

output = []
prev_time = -1

for x, t, label in zip(features, times, labels):
    global_score = global_scorer.score(x)
    adaptive_score = adaptive_scorer.score(x)
    del_t = 0 if prev_time == -1 else t - prev_time
    if label == 0:
        adaptive_scorer.update(x, del_t)
    prev_time = t
    output.append([t, label, adaptive_score, global_score])





### Collecting results

Combine the per-transaction scores into a single DataFrame with global and adaptive scores per transaction, then convert to a csv for use later on.

In [3]:
columns = ['Time', 'Class', 'Adaptive Score', 'Global Score']

findings = pd.DataFrame(output, columns=columns)

filepath = '../data/findings.csv'

findings.to_csv(filepath, index=False)